In [11]:
# TI4 Factions classification with auto_sklearn2 + cross-validation (no train/test split)
# Predict target: Winner (whether the faction won their game)
from auto_sklearn2 import AutoSklearnClassifier
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score

# -----------------------------
# Config
# -----------------------------
k = 3             # number of CV folds
time_limit = 120  # seconds per AutoML fit (per fold)
random_state = 42 # reproducibility for KFold shuffling

In [12]:
TRANSFORM_URL = 'https://raw.githubusercontent.com/Oldmage112/TI4-Project/refs/heads/main/PowerBI/Derived%20CSVs/Factions_Transform.csv'
WINNERS_URL   = 'https://raw.githubusercontent.com/Oldmage112/TI4-Project/refs/heads/main/PowerBI/Derived%20CSVs/Factions_Winners.csv'

df_transform = pd.read_csv(TRANSFORM_URL, index_col=-1)
df_winners   = pd.read_csv(WINNERS_URL, index_col=0)

# Merge on Index so each row has both feature and label
df = df_transform.merge(df_winners, on='Index', how='inner')
df.index = df.iloc[:,0]
df = df.drop(columns='Index')

# Target: 'Winner' column from Factions_Winners
# Adjust target_col if the winners CSV uses a different column name
target_col = 'WinningFaction'
print(df.shape)
df.head()

(17797, 49)


,Points,Expansions,PlayerCount,Platform,A Sickening Lurch,Arborec,Argent Flight,Avarice Rex,Barony of Letnev,Clan of Saar,...,Ruby Monarch,Saint of Swords,Sardakk N'orr,Titans of Ul,Universities of Jol-Nar,Vuil'raith Cabal,Winnu,Xxcha Kingdom,Yin Brotherhood,WinningFaction
Index,,,,,,,,,,,,,,,,,,,,,
396,10,Prophecy of Kings Only,6,Async,0,0,1,0,0,0,...,0,0,1,0,1,1,0,1,0,XxchaKingdom
600,10,Prophecy of Kings Only,6,Async,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,KeleresMentak
831,10,Prophecy of Kings Only,6,Async,0,0,1,0,0,0,...,0,0,0,1,0,1,0,0,0,YssarilTribes
3184,10,Prophecy of Kings Only,6,Async,0,0,1,0,1,0,...,0,0,0,1,1,1,1,0,0,ArgentFlight
3252,10,Prophecy of Kings Only,6,Async,0,0,1,0,0,0,...,0,0,0,0,0,1,0,1,1,Vuil'raithCabal


In [13]:
# -----------------------------
# 3) Encode features & label-encode the target
# -----------------------------
feature_cols = df.drop(columns=[target_col])
numeric_features = feature_cols.select_dtypes(include=np.number).columns
categorical_features = feature_cols.select_dtypes(exclude=np.number).columns
numeric_features = feature_cols.loc[:,['Points','PlayerCount']].columns
categorical_features = feature_cols.loc[:,['Points','PlayerCount','Expansions', 'Platform', 'A Sickening Lurch','Arborec', 'Argent Flight', 'Avarice Rex', 'Barony of Letnev'
                 ,'Clan of Saar', 'Crimson Rebellion', 'Deepwrought Scholarate'
                 ,'El Nen Janovet', 'Embers of Muaat', 'Emirates of Hacan', 'Empyrean'
                 ,'Expansions_y', 'Federation of Sol', 'Firmament', 'Ghosts of Creuss'
                 ,'Il Na Viroset', 'Il Sai Lakoe, Herald of Thorns', 'Keleres',
       'Keleres Argent', 'Keleres Mentak', 'Keleres Xxcha', 'L1Z1X Mindnet',
       'Last Bastion', 'Mahact Gene-Sorcerers', 'Mentak Coalition',
       'Naalu Collective', 'Naaz-Rokha Alliance', 'Nekro Virus', 'Nomad',
       'Obsidian', 'Platform_y', 'PlayerCount_y', 'Radiant Aur',
       'Ral Nel Consortium', 'Ruby Monarch', 'Saint of Swords',
       "Sardakk N'orr", 'Titans of Ul', 'Universities of Jol-Nar',
       "Vuil'raith Cabal", 'Winnu', 'Xxcha Kingdom', 'Yin Brotherhood']].columns
numeric_features = []

df_enc = pd.get_dummies(df, columns=categorical_features, drop_first=True)

# Label-encode the target (species -> 0, 1, 2)
species_cat = pd.Categorical(df_enc[target_col])
label_map = dict(enumerate(species_cat.categories))  # {0: 'Adelie', 1: 'Chinstrap', 2: 'Gentoo'}
print("Label mapping:", label_map)

# Split X / y (no holdout; we will do CV over the whole dataset)
X = df_enc.drop(columns=[target_col]).reset_index(drop=True)
y = pd.Series(species_cat.codes, name=target_col).reset_index(drop=True)

feature_cols.loc[:,['Points', 'PlayerCount', 'A Sickening Lurch', 'Arborec']].columns

feature_cols.columns
feature_cols.loc[:,['Points','PlayerCount']].columns
feature_cols.loc[:,['Points', 'Expansions', 'PlayerCount', 'Platform', 'A Sickening Lurch','Arborec', 'Argent Flight', 'Avarice Rex', 'Barony of Letnev'
                 ,'Clan of Saar', 'Crimson Rebellion', 'Deepwrought Scholarate'
                 ,'El Nen Janovet', 'Embers of Muaat', 'Emirates of Hacan', 'Empyrean'
                 ,'Expansions_y', 'Federation of Sol', 'Firmament', 'Ghosts of Creuss'
                 ,'Il Na Viroset', 'Il Sai Lakoe, Herald of Thorns', 'Keleres',
       'Keleres Argent', 'Keleres Mentak', 'Keleres Xxcha', 'L1Z1X Mindnet',
       'Last Bastion', 'Mahact Gene-Sorcerers', 'Mentak Coalition',
       'Naalu Collective', 'Naaz-Rokha Alliance', 'Nekro Virus', 'Nomad',
       'Obsidian', 'Platform_y', 'PlayerCount_y', 'Radiant Aur',
       'Ral Nel Consortium', 'Ruby Monarch', 'Saint of Swords',
       "Sardakk N'orr", 'Titans of Ul', 'Universities of Jol-Nar',
       "Vuil'raith Cabal", 'Winnu', 'Xxcha Kingdom', 'Yin Brotherhood']].columns


Label mapping: {0: 'ASickeningLurch', 1: 'Arborec', 2: 'ArgentFlight', 3: 'AvariceRex', 4: 'BaronyofLetnev', 5: 'ClanofSaar', 6: 'CrimsonRebellion', 7: 'DeepwroughtScholarate', 8: 'ElNenJanovet', 9: 'EmbersofMuaat', 10: 'EmiratesofHacan', 11: 'Empyrean', 12: 'FederationofSol', 13: 'Firmament', 14: 'GhostsofCreuss', 15: 'IlNaViroset', 16: 'IlSaiLakoeHeraldofThorns', 17: 'Keleres', 18: 'KeleresArgent', 19: 'KeleresMentak', 20: 'KeleresXxcha', 21: 'L1Z1XMindnet', 22: 'LastBastion', 23: 'MahactGeneSorcerers', 24: 'MentakCoalition', 25: 'NaaluCollective', 26: 'NaazRokhaAlliance', 27: 'NekroVirus', 28: 'Nomad', 29: 'RadiantAur', 30: 'RalNelConsortium', 31: 'RubyMonarch', 32: 'SaintofSwords', 33: "SardakkN'orr", 34: 'TitansofUl', 35: 'UniversitiesofJolNar', 36: "Vuil'raithCabal", 37: 'Winnu', 38: 'XxchaKingdom', 39: 'YinBrotherhood', 40: 'YssarilTribes'}


Index(['Points', 'Expansions', 'PlayerCount', 'Platform', 'A Sickening Lurch',
       'Arborec', 'Argent Flight', 'Avarice Rex', 'Barony of Letnev',
       'Clan of Saar', 'Crimson Rebellion', 'Deepwrought Scholarate',
       'El Nen Janovet', 'Embers of Muaat', 'Emirates of Hacan', 'Empyrean',
       'Expansions_y', 'Federation of Sol', 'Firmament', 'Ghosts of Creuss',
       'Il Na Viroset', 'Il Sai Lakoe, Herald of Thorns', 'Keleres',
       'Keleres Argent', 'Keleres Mentak', 'Keleres Xxcha', 'L1Z1X Mindnet',
       'Last Bastion', 'Mahact Gene-Sorcerers', 'Mentak Coalition',
       'Naalu Collective', 'Naaz-Rokha Alliance', 'Nekro Virus', 'Nomad',
       'Obsidian', 'Platform_y', 'PlayerCount_y', 'Radiant Aur',
       'Ral Nel Consortium', 'Ruby Monarch', 'Saint of Swords',
       'Sardakk N'orr', 'Titans of Ul', 'Universities of Jol-Nar',
       'Vuil'raith Cabal', 'Winnu', 'Xxcha Kingdom', 'Yin Brotherhood'],
      dtype='str')

In [14]:
y

0        38
1        19
2        40
3         2
4        36
         ..
17792    11
17793    37
17794    23
17795    23
17796    23
Name: WinningFaction, Length: 17797, dtype: int8

In [15]:
# -----------------------------
# 4) K-fold cross-validation over the entire dataset
# We train a fresh AutoSklearnClassifier in each fold and compute OOF metrics.
# -----------------------------
kf = KFold(n_splits=k, shuffle=True)  # , random_state=random_state

cv_acc = []
cv_f1 = []

# Out-of-fold predictions container (same order as y)
oof_pred = np.full(shape=len(y), fill_value=-1, dtype=int)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    # A smaller time budget per fold keeps total runtime bounded
    automl_cv = AutoSklearnClassifier(time_limit=time_limit)
    automl_cv.fit(X_tr, y_tr)

    y_va_pred = automl_cv.predict(X_va)
    oof_pred[va_idx] = y_va_pred

    acc = accuracy_score(y_va, y_va_pred)
    f1  = f1_score(y_va, y_va_pred, average="macro")
    cv_acc.append(acc)
    cv_f1.append(f1)
    print(f"[Fold {fold}] Accuracy={acc:.4f} | F1 (macro)={f1:.4f}")

# Summary across folds
print(f"\n=== Cross-Validation Summary ({k}-fold) ===")
print(f"Accuracy Mean: {np.mean(cv_acc):.4f} | Accuracy Std: {np.std(cv_acc):.4f}")
print(f"F1 (macro) Mean: {np.mean(cv_f1):.4f} | F1 (macro) Std: {np.std(cv_f1):.4f}")

# Out-of-fold performance: single estimate over all rows using OOF predictions
oof_acc = accuracy_score(y, oof_pred)
oof_f1  = f1_score(y, oof_pred, average="macro")
print(f"\n=== Out-of-Fold (OOF) Performance on Full Dataset ===")
print(f"OOF Accuracy: {oof_acc:.4f} | OOF F1 (macro): {oof_f1:.4f}")

c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is

[Fold 1] Accuracy=0.2173 | F1 (macro)=0.1629


c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is

[Fold 2] Accuracy=0.2190 | F1 (macro)=0.1542


c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is

[Fold 3] Accuracy=0.2218 | F1 (macro)=0.1715

=== Cross-Validation Summary (3-fold) ===
Accuracy Mean: 0.2194 | Accuracy Std: 0.0019
F1 (macro) Mean: 0.1629 | F1 (macro) Std: 0.0071

=== Out-of-Fold (OOF) Performance on Full Dataset ===
OOF Accuracy: 0.2194 | OOF F1 (macro): 0.1685


In [9]:
# -----------------------------
# 5) Fit final model on the full dataset (for deployment)
# No holdout; you rely on CV/OOF for generalization estimates.
# -----------------------------
auto_sklearn_full = AutoSklearnClassifier(time_limit=time_limit)
auto_sklearn_full.fit(X, y)

# Try to report best params if available
best_params = getattr(auto_sklearn_full, "best_params", None)
if best_params is None:
    best_params = getattr(auto_sklearn_full, "best_params_", "N/A")

print("\n=== Final Model Trained on Full Data ===")
print(f"Best params: {best_params}")
print(f"Label mapping (for decoding predictions): {label_map}")

c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\MGarn\anaconda3\envs\autoskl2\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



=== Final Model Trained on Full Data ===
Best params: {'preprocessor': 'standard_scaler', 'classifier': 'logistic_regression'}
Label mapping (for decoding predictions): {0: 'ASickeningLurch', 1: 'Arborec', 2: 'ArgentFlight', 3: 'AvariceRex', 4: 'BaronyofLetnev', 5: 'ClanofSaar', 6: 'CrimsonRebellion', 7: 'DeepwroughtScholarate', 8: 'ElNenJanovet', 9: 'EmbersofMuaat', 10: 'EmiratesofHacan', 11: 'Empyrean', 12: 'FederationofSol', 13: 'Firmament', 14: 'GhostsofCreuss', 15: 'IlNaViroset', 16: 'IlSaiLakoeHeraldofThorns', 17: 'Keleres', 18: 'KeleresArgent', 19: 'KeleresMentak', 20: 'KeleresXxcha', 21: 'L1Z1XMindnet', 22: 'LastBastion', 23: 'MahactGeneSorcerers', 24: 'MentakCoalition', 25: 'NaaluCollective', 26: 'NaazRokhaAlliance', 27: 'NekroVirus', 28: 'Nomad', 29: 'RadiantAur', 30: 'RalNelConsortium', 31: 'RubyMonarch', 32: 'SaintofSwords', 33: "SardakkN'orr", 34: 'TitansofUl', 35: 'UniversitiesofJolNar', 36: "Vuil'raithCabal", 37: 'Winnu', 38: 'XxchaKingdom', 39: 'YinBrotherhood', 40: 'Y

In [10]:
# Optional: show leaderboard if provided by auto_sklearn2
if hasattr(auto_sklearn_full, "get_models_performance"):
    print("\nModel leaderboard (auto_sklearn2):")
    perf = auto_sklearn_full.get_models_performance()
    df_lb = pd.DataFrame(list(perf.items()), columns=["model", "score"])
    df_lb.sort_values("score", ascending=False, inplace=True)
    print(df_lb)
else:
    print("\n(Leaderboard API not available in this version.")


Model leaderboard (auto_sklearn2):
                                 model     score
2  standard_scaler_logistic_regression  0.230207
0        standard_scaler_random_forest  0.200652
1    standard_scaler_gradient_boosting  0.174800
